In [1]:
import pandas as pd
import numpy as np

url = ("https://raw.githubusercontent.com/jxchen/Kaggle"
       "/master/Give%20Me%20Some%20Credit/cs-training.csv")
df_raw = pd.read_csv(url, index_col=0)
df = df_raw.rename(columns={
    "SeriousDlqin2yrs":                    "vo_no",
    "RevolvingUtilizationOfUnsecuredLines": "ty_le_su_dung_tin_dung",
    "age":                                  "tuoi",
    "NumberOfTime30-59DaysPastDueNotWorse": "so_lan_tre_30_59_ngay",
    "DebtRatio":                            "ty_le_no",
    "MonthlyIncome":                        "thu_nhap_thang",
    "NumberOfOpenCreditLinesAndLoans":      "so_tai_khoan_vay",
    "NumberOfTimes90DaysLate":              "so_lan_tre_90_ngay",
    "NumberRealEstateLoansOrLines":         "so_tai_khoan_bat_dong_san",
    "NumberOfTime60-89DaysPastDueNotWorse": "so_lan_tre_60_89_ngay",
    "NumberOfDependents":                   "so_nguoi_phu_thuoc"
})
print(f"Shape: {df.shape} | NaN: {df.isnull().sum().sum()}")

Shape: (150000, 11) | NaN: 33655


In [2]:
col = df["thu_nhap_thang"]

# Tính Q1, Q3, IQR — giống tính tay
Q1  = col.quantile(0.25)
Q3  = col.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1={Q1:,.0f} | Q3={Q3:,.0f} | IQR={IQR:,.0f}")
print(f"Ngưỡng IQR: [{lower:,.0f}, {upper:,.0f}]")

# Đếm outlier
n_outlier = ((col < lower) | (col > upper)).sum()
print(f"Số outlier: {n_outlier:,} ({n_outlier/len(col):.1%})")

Q1=3,400 | Q3=8,249 | IQR=4,849
Ngưỡng IQR: [-3,874, 15,522]
Số outlier: 4,879 (3.3%)


In [3]:
# Tính Z-score cho toàn cột
col_clean = col.dropna()  # bỏ NaN trước khi tính

mean = col_clean.mean()
std  = col_clean.std(ddof=1)  # ddof=1 vì đây là mẫu

zscore = (col_clean - mean) / std

# Đếm outlier theo Z-score
n_zscore = (zscore.abs() > 3).sum()
print(f"Outlier Z-score: {n_zscore:,} ({n_zscore/len(col_clean):.1%})")

# So sánh hai phương pháp
print(f"\nIQR phát hiện:     {n_outlier:,} outlier")
print(f"Z-score phát hiện: {n_zscore:,} outlier")
print("→ Cái nào nhiều hơn? Tại sao?")

Outlier Z-score: 321 (0.3%)

IQR phát hiện:     4,879 outlier
Z-score phát hiện: 321 outlier
→ Cái nào nhiều hơn? Tại sao?


In [6]:
# Thay mã lỗi 96/98 → NaN (đúng với domain knowledge)
cols_tre_han = ["so_lan_tre_30_59_ngay",
                "so_lan_tre_60_89_ngay",
                "so_lan_tre_90_ngay"]

for col in cols_tre_han:
    n_loi = df[col].isin([96, 98]).sum()
    print(f"{col}: {n_loi:,} mã lỗi")
    df[col] = df[col].replace([96, 98], np.nan)

# Xử lý tuoi = 0
n_tuoi_0 = (df["tuoi"] == 0).sum()
print(f"\ntuoi = 0: {n_tuoi_0} hàng")
df.loc[df["tuoi"] == 0, "tuoi"] = np.nan

# Clip ty_le_su_dung — lỗi kỹ thuật rõ ràng
print(f"\nMax trước clip: {df['ty_le_su_dung_tin_dung'].max():,.0f}")
df["ty_le_su_dung_tin_dung"] = df["ty_le_su_dung_tin_dung"].clip(upper=2)
print(f"Max sau clip:   {df['ty_le_su_dung_tin_dung'].max():,.2f}")

so_lan_tre_30_59_ngay: 0 mã lỗi
so_lan_tre_60_89_ngay: 0 mã lỗi
so_lan_tre_90_ngay: 0 mã lỗi

tuoi = 0: 0 hàng

Max trước clip: 2
Max sau clip:   2.00


In [7]:
mask_loi = (df["so_lan_tre_30_59_ngay"].isna() &
            df["so_lan_tre_60_89_ngay"].isna() &
            df["so_lan_tre_90_ngay"].isna())
print(mask_loi.sum())  # có phải đúng 269 không?

269


In [8]:
# BƯỚC 1: Chia train/test TRƯỚC — rồi mới xử lý
train = df.sample(frac=0.8, random_state=42)
test  = df.drop(train.index)
train = train.copy()
test  = test.copy()

# BƯỚC 2: Fit trên TRAIN — transform cả hai
# Gợi ý: median thu_nhap, median so_nguoi_phu_thuoc
# Điền NaN đúng cách

# BƯỚC 3: Verify
print(f"Train NaN còn lại: {train.isnull().sum().sum()}")
print(f"Test  NaN còn lại: {test.isnull().sum().sum()}")
print(f"Train shape: {train.shape}")
print(f"Test  shape: {test.shape}")

Train NaN còn lại: 27637
Test  NaN còn lại: 6826
Train shape: (120000, 11)
Test  shape: (30000, 11)


In [4]:
train = df.sample(frac=0.8, random_state=42)
test  = df.drop(train.index)
train = train.copy()
test  = test.copy()

median_thu_nhap = train["thu_nhap_thang"].median()
median_so_nguoi_phu_thuoc = train["so_nguoi_phu_thuoc"].median()

#Điền NaN
train["thu_nhap_thang"] = train["thu_nhap_thang"].fillna(median_thu_nhap)
train["so_nguoi_phu_thuoc"] = train["so_nguoi_phu_thuoc"].fillna(median_so_nguoi_phu_thuoc)

test["thu_nhap_thang"] = test["thu_nhap_thang"].fillna(median_thu_nhap)
test["so_nguoi_phu_thuoc"] = test["so_nguoi_phu_thuoc"].fillna(median_so_nguoi_phu_thuoc)

#verify
print(f"Train NaN còn lại: {train.isnull().sum().sum()}")
print(f"Test  NaN còn lại: {test.isnull().sum().sum()}")
print(f"Train shape: {train.shape}")
print(f"Test  shape: {test.shape}")

Train NaN còn lại: 0
Test  NaN còn lại: 0
Train shape: (120000, 11)
Test  shape: (30000, 11)
